# 深度学习作业 HW04

姓名：邱宇
学号：20234080317

---

# 第2章：序列模型

## 2.1 理论计算题

序列：ababc
全部字符：a b a b c
原始转移计数（前一字符→当前字符）
a→b：2 次
b→a：1 次
b→c：1 次
其余转移出现次数为 0
词汇表 V = {a,b,c}，词数量 | V|=3
拉普拉斯平滑公式：
p (y|x) = count (x→y) + 1 / (sum_{所有 y'} count (x→y') + |V|)
求 p (a|b)
第一步：统计以 b 为前驱的所有转移总数
b 后面出现 a、c，总次数 = 1+1=2
count (b→a)=1
分子 = 1 + 1 = 2
分母 = 2 + 3 = 5
p (a|b) = 2/5
求 p (c|b)
count (b→c)=1
分子 = 1 + 1 = 2
分母同上 = 5
p (c|b) = 2/5

## 2.2 编程题：文本预处理函数

In [ ]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    文本预处理函数
    参数：
        text: 输入文本字符串
        n: n-gram的n值
    返回：
        vocab: 词汇表（字典，词->索引）
        (特征列表, 标签列表)
    """
    # 1. 转为小写，去除标点符号（保留单词和空格）
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(words)
    # 按频率降序排列，频率相同的按字母顺序
    sorted_words = sorted(word_counts.keys(), key=lambda x: (-word_counts[x], x))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 4. 构建n-gram特征和标签
    features = []
    labels = []
    for i in range(len(words) - n):
        feature = words[i:i+n]
        label = words[i+n] if i+n < len(words) else None
        features.append(feature)
        labels.append(label)
    
    return vocab, (features, labels)

# 测试
text = "The time machine"
vocab, (features, labels) = preprocess_text(text, n=2)
print("=== 文本预处理测试 ===")
print(f"词汇表: {vocab}")
print(f"特征 (n-grams): {features}")
print(f"标签: {labels}")
print(f"预期特征: [['the', 'time'], ['time', 'machine']]")
print(f"预期标签: ['machine', None]")

---

# 第3章：循环神经网络

## 3.1 理论计算题

模型定义：
无偏置 RNN：h_t = W_hh * h_{t-1} + W_hx * x_t
输出：o_t = W_oh * h_t
损失 L = 0.5 * sum_{t=1}^T (o_t - y_t)^2
1. 梯度推导 dL/dW_hh
链式法则：
dL/dW_hh = sum_{t=1}^T [ dL/dh_t * dh_t/dW_hh ]
dh_t/dW_hh = h_{t-1} + W_hh * dh_{t-1}/dW_hh
令 delta_t = dL/dh_t
delta_t = dL/do_t * do_t/dh_t + delta_{t+1} * dh_{t+1}/dh_t
dL/do_t = o_t - y_t
do_t/dh_t = W_oh
dh_{t+1}/dh_t = W_hh
所以 delta_t = (o_t - y_t) W_oh + delta_{t+1} W_hh
边界条件：delta_{T+1}=0
dh_t/dW_hh = sum_{k=1}^{t} (product_{s=k+1}^{t} W_hh) h_{k-1}
整体梯度：
dL/dW_hh = sum_{t=1}^T delta_t * [ sum_{k=1}^{t} (W_hh)^(t-k) h_{k-1} ]
展开时间步：
dL/dW_hh = sum_{t=1}^T sum_{k=1}^t [ (product_{s=k+1}^t W_hh) * delta_t h_{k-1} ]
2. 梯度消失 / 爆炸条件
梯度递推中存在连续矩阵乘积 W_hh^(T-k)，取决于 W_hh 的谱半径（最大特征值绝对值 λ）：
梯度消失：λ < 1，多层矩阵相乘后数值指数衰减，远距离时间步梯度趋近 0
梯度爆炸：λ > 1，多层矩阵相乘后数值指数增大，梯度数值无限变大

## 3.2 编程题：实现RNN单元

In [ ]:
import torch
import torch.nn as nn

class RNNCell(nn.Module):
    def __init__(self, input_size, hidden_size, bias=True):
        super(RNNCell, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # 权重矩阵
        self.W_hx = nn.Parameter(torch.randn(hidden_size, input_size))
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size))
        self.b_h = nn.Parameter(torch.zeros(hidden_size)) if bias else None
    
    def forward(self, x_t, h_prev):
        """
        前向传播
        参数：
            x_t: 输入 (batch_size, input_size)
            h_prev: 上一时刻隐藏状态 (batch_size, hidden_size)
        返回：
            h_t: 当前隐藏状态 (batch_size, hidden_size)
        """
        # 计算当前隐藏状态：h_t = tanh(W_hh * h_prev + W_hx * x_t + b_h)
        h_t = torch.tanh(torch.matmul(h_prev, self.W_hh.t()) + 
                         torch.matmul(x_t, self.W_hx.t()) + 
                         self.b_h)
        return h_t
    
    def backward(self, x_t, h_prev, h_t, dh_next):
        """
        反向传播（仅计算梯度，不更新参数）
        参数：
            x_t: 输入 (batch_size, input_size)
            h_prev: 上一时刻隐藏状态 (batch_size, hidden_size)
            h_t: 当前隐藏状态 (batch_size, hidden_size)
            dh_next: 来自下一时刻的梯度 (batch_size, hidden_size)
        返回：
            dx_t: 输入梯度
            dh_prev: 隐藏状态梯度
            dW_hx: 输入权重梯度
            dW_hh: 隐藏权重梯度
            db_h: 偏置梯度
        """
        batch_size = x_t.size(0)
        
        # tanh的导数：d_tanh(x) = 1 - tanh^2(x)
        dh_raw = (1 - h_t ** 2)
        
        # 总梯度（来自当前输出损失 + 来自下一时刻的梯度）
        dh = dh_next * dh_raw
        
        # 计算各参数梯度
        dW_hh = torch.matmul(dh.t(), h_prev)
        dW_hx = torch.matmul(dh.t(), x_t)
        db_h = dh.sum(dim=0)
        
        # 计算输入梯度
        dx_t = torch.matmul(dh, self.W_hx)
        dh_prev = torch.matmul(dh, self.W_hh)
        
        return dx_t, dh_prev, dW_hx, dW_hh, db_h

print("=== RNN单元测试 ===")

batch_size = 2
input_size = 10
hidden_size = 5

torch.manual_seed(42)

rnn_cell = RNNCell(input_size, hidden_size)

x_t = torch.randn(batch_size, input_size)
h_prev = torch.randn(batch_size, hidden_size)

# 前向传播
h_t = rnn_cell(x_t, h_prev)
print(f"输入形状: {x_t.shape}")
print(f"上一隐藏状态形状: {h_prev.shape}")
print(f"当前隐藏状态形状: {h_t.shape}")
print(f"前向传播测试通过: {h_t.shape == (batch_size, hidden_size)}")

# 反向传播
dh_next = torch.randn(batch_size, hidden_size)
dx_t, dh_prev_grad, dW_hx, dW_hh, db_h = rnn_cell.backward(x_t, h_prev, h_t, dh_next)
print(f"\n反向传播梯度形状:")
print(f"  dx_t: {dx_t.shape}")
print(f"  dh_prev: {dh_prev_grad.shape}")
print(f"  dW_hx: {dW_hx.shape}")
print(f"  dW_hh: {dW_hh.shape}")
print(f"  db_h: {db_h.shape}")
print("反向传播测试通过!")

---

# 第4章：高级循环神经网络

## 4.1 理论计算题

已知：
L 层，每层隐藏单元 H，输入维度 D，输出维度 O
双向：每层分前向、后向两个独立 RNN
单层单向 RNN 参数（权重 + 偏置）：
输入到隐藏：W_hx (D×H) + b_h (H)
隐藏自循环：W_hh (H×H) + b_hh (H)
单层单向参数总量 = DH + H + HH + H = HD + HH + 2H
单层双向 = 2 * (HD + HH + 2H)
L 层双向总循环层参数 = L * 2 * (H*D + H² + 2H)
最后输出层（拼接前向 + 后向隐藏，维度 2H）：
W_out (2H × O) + b_out (O)，参数数量 = 2H*O + O
全部参数总和表达式：
Total = 2 * L * (DH + HH + 2H) + 2H*O + O
5.1 Skip-gram 负采样损失函数
符号定义：
v_c：中心词输入向量
u_o：正上下文输出向量
u_nk：第 k 个负样本输出向量，共 K 个负样本
sigmoid 函数 σ(z) = 1/(1+exp (-z))

## 4.2 编程题：实现双向RNN编码器

In [ ]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(BidirectionalRNNEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.rnn = nn.RNN(input_dim, hidden_dim, batch_first=True, bidirectional=True)
    
    def forward(self, X):
        """
        参数：
            X: 输入序列 (seq_len, batch, input_dim)
        返回：
            outputs: 每个时间步的拼接前后向隐藏状态 (seq_len, batch, 2*hidden_dim)
            final_state: 最终隐藏状态（作为序列表示）
        """
        # 前向传播
        outputs, (h_n) = self.rnn(X)
        
        # outputs: (seq_len, batch, 2*hidden_dim)
        # h_n: (2, batch, hidden_dim) - 2个方向的最后隐藏状态
        
        # 获取最终隐藏状态
        # 前向最后隐藏状态 h_n[0] 和后向最后隐藏状态 h_n[1]
        # 拼接作为序列表示
        h_forward = h_n[0]   # (batch, hidden_dim)
        h_backward = h_n[1]  # (batch, hidden_dim)
        final_state = torch.cat([h_forward, h_backward], dim=1)  # (batch, 2*hidden_dim)
        
        return outputs, final_state

print("=== 双向RNN编码器测试 ===")

seq_len = 5
batch = 3
input_dim = 10
hidden_dim = 4

encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)

# 输入形状: (seq_len, batch, input_dim)
X = torch.randn(seq_len, batch, input_dim)

outputs, final_state = encoder(X)

print(f"输入形状: {X.shape}")
print(f"每个时间步输出形状: {outputs.shape}")
print(f"预期形状: ({seq_len}, {batch}, {2*hidden_dim})")
print(f"最终隐藏状态形状: {final_state.shape}")
print(f"预期形状: ({batch}, {2*hidden_dim})")

print("\n测试结果:")
print(f"  前向传播: {'通过' if outputs.shape == (seq_len, batch, 2*hidden_dim) else '失败'}")
print(f"  隐藏状态: {'通过' if final_state.shape == (batch, 2*hidden_dim) else '失败'}")

---

# 第5章：嵌入向量

## 5.1 理论计算题

符号定义：
v_c：中心词输入向量
u_o：正上下文输出向量
u_nk：第 k 个负样本输出向量，共 K 个负样本
sigmoid 函数 σ(z) = 1/(1+exp (-z))
1. 对数似然目标函数
单个 (中心词，上下文) 样本损失（最大化对数似然，等价最小化负对数）：
L = - [ log (σ(v_c・u_o)) + sum_{k=1}^K log (1 - σ(v_c・u_{n_k})) ]
完整批量目标：对所有窗口内所有中心 - 上下文对求和上述 L
2. 负样本采样方式
噪声分布通常采用词频 3/4 次方分布（unigram 分布）：P (w) = count (w)^(3/4) /sum_{w'} count (w')^(3/4)
根据该分布随机抽取 K 个非当前正上下文的单词作为负样本；允许重复采样，不排除中心词

## 5.2 编程题：实现CBOW模型

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(CBOW, self).__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        
        # 嵌入层
        self.W = nn.Parameter(torch.randn(vocab_size, embedding_dim))
        # 输出权重
        self.W_out = nn.Parameter(torch.randn(embedding_dim, vocab_size))
    
    def forward(self, context_indices, target_index):
        """
        参数：
            context_indices: 上下文词的索引 (batch, context_size * 2)
            target_index: 目标词的索引 (batch,)
        返回：
            loss: 交叉熵损失
        """
        batch_size = context_indices.size(0)
        
        # 1. 获取上下文词的嵌入向量
        context_embeddings = self.W[context_indices]  # (batch, context_size*2, d)
        
        # 2. 计算上下文词的平均值（作为隐藏层）
        hidden = context_embeddings.mean(dim=1)  # (batch, d)
        
        # 3. 计算输出分数
        logits = torch.matmul(hidden, self.W_out)  # (batch, V)
        
        # 4. 计算交叉熵损失（目标为中心词索引）
        loss = F.cross_entropy(logits, target_index)
        
        return loss

print("=== CBOW模型测试 ===")

batch_size = 4
vocab_size = 100
embedding_dim = 50
context_size = 2  # 上下文中词的数量（一边）

torch.manual_seed(42)

cbow = CBOW(vocab_size, embedding_dim)

# 模拟输入
# context_indices: (batch, context_size * 2)，每个样本有context_size*2个上下文词
context_indices = torch.randint(0, vocab_size, (batch_size, context_size * 2))
# target_index: (batch,)，目标词索引
target_index = torch.randint(0, vocab_size, (batch_size,))

print(f"上下文索引形状: {context_indices.shape}")
print(f"目标词索引形状: {target_index.shape}")

# 前向传播
loss = cbow(context_indices, target_index)
print(f"损失值: {loss.item():.4f}")

# 验证损失计算
print(f"\n模型参数:")
print(f"  W形状: {cbow.W.shape}")
print(f"  W_out形状: {cbow.W_out.shape}")
print(f"  隐藏层形状: ({batch_size}, {embedding_dim})")
print(f"  输出形状: ({batch_size}, {vocab_size})")
print("\n测试通过!")

---

# 第6章：注意力机制

## 6.1 理论计算题

已知：
Q ∈ R (2×4), K ∈ R (3×4), V ∈ R (3×5)
dk = 4，缩放系数 sqrt (dk) = 2
步骤 1：计算 Q 与 K 转置的点积得分矩阵 Score = Q @ K.T
Score 形状：2 行 3 列
步骤 2：缩放得分 Scaled_Score = Score /sqrt (4) = Score / 2
步骤 3：对 Scaled_Score 每行做 softmax，得到注意力权重 Attn_Weight，每行和为 1，形状 2×3
步骤 4：输出 Output = Attn_Weight @ V，最终输出矩阵形状 2×5
通用符号计算过程
K 转置 K.T ∈ R (4×3)
Score(2,3) = Q(2,4) · K.T(4,3)
Score[i,j] = sum_{d=1~4} Q[i,d] * K[j,d]
ScaledScore[i,j] = Score[i,j] / 2
AttnWeight[i,j] = exp(ScaledScore[i,j]) / sum_{j'=1~3} exp(ScaledScore[i,j'])
Output [i,m] = sum_{j=1~3} AttnWeight [i,j] * V [j,m]
输出维度：2 × 5

## 6.2 编程题：实现Multi-Head Attention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度
        
        # 线性变换矩阵
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, X):
        """
        参数：
            X: 输入 (seq_len, batch, d_model)
        返回：
            output: 输出 (seq_len, batch, d_model)
        """
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性变换得到 Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)
        V = self.W_v(X)
        
        # 2. 分头：重塑为 (seq_len, batch, num_heads, d_k) 然后转置
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(1, 2)  # (seq_len, num_heads, batch, d_k)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. 计算注意力分数
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (seq_len, num_heads, batch, seq_len)
        
        # 4. Softmax
        attn_weights = F.softmax(scores, dim=-1)
        
        # 5. 加权求和
        attn_output = torch.matmul(attn_weights, V)  # (seq_len, num_heads, batch, d_k)
        
        # 6. 合并多头：转置并reshape
        attn_output = attn_output.transpose(1, 2).contiguous().view(seq_len, batch_size, self.d_model)
        
        # 7. 最终线性变换
        output = self.W_o(attn_output)
        
        return output

print("=== Multi-Head Attention 测试 ===")

seq_len = 5
batch = 2
d_model = 4
num_heads = 2

torch.manual_seed(42)

X = torch.randn(seq_len, batch, d_model)
print(f"输入形状: {X.shape}")
print(f"num_heads: {num_heads}, d_k: {d_model // num_heads}")

mha = MultiHeadAttention(d_model, num_heads)
output = mha(X)

print(f"输出形状: {output.shape}")
print(f"预期形状: ({seq_len}, {batch}, {d_model})")
print(f"测试: {'通过' if output.shape == (seq_len, batch, d_model) else '失败'}")

# 验证各部分形状
print(f"\n各部分验证:")
print(f"  d_k = {d_model // num_heads}")
print(f"  输出维度与输入维度一致: {output.shape == X.shape}")